<a href="https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane: Refresh / Content Opportunity Scoring**

Task type: **Classification** (primary) with a **ranking/scoring** layer on top. The core prediction is binary — will this page decline (or stay a review candidate) or not but the actual product isn't the label itself, it's an ordered queue: pages sorted by predicted decline probability, so a reviewer works top-down under limited capacity. This matches Week 2's structure: a decision tree classifier producing `predict_proba`, then ranked by that probability and scored with Precision@K.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/viki22uied/ML"
REPO_DIR = "ML"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())
print(os.listdir())


Now in: /content/ML/ML/ML/ML
['data', 'GUIDE.md', 'notebooks', 'submission', 'skills', 'DATA_USE.md', '.github', '.git', 'scripts', 'SETUP.md', 'requirements.txt', 'CLAUDE.md', 'outputs', 'LICENSE', 'README.md', 'docs', 'AGENTS.md', '.gitignore', 'work']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target (proxy): `is_declining_label = (trend_direction == "down")`, the same proxy used in the starter pipeline.

This is a **current-window proxy**, not a genuine future outcome — it's a bucket calculated from the current trend, not "will this page decline over the next 30 days." A stronger target for later weeks would be a forward-looking label: features from a prior window (e.g. 90 days) predicting decline over a future window (e.g. next 30 days), which avoids the risk of the label partially encoding the same information as the features.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@K** (K = weekly reviewer capacity, e.g. 20 or 50), not accuracy or raw AUC.

This matches the real decision: a reviewer only looks at the top K pages the model ranks first, so what matters is how many of *those* are genuinely worth reviewing not how well the model classifies all 30,000 pages. Week 2 showed this concretely: a depth-2 tree scored 0.550/0.600 (Precision@20/@50) in-sample but only 0.400/0.520 on client holdout — accuracy alone would have hidden that gap.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 20
print(f"Reviewer capacity per week (K): {K} pages")


Reviewer capacity per week (K): 20 pages


## 4. The unit of analysis, as a real dataframe

One row = one page (`content_id`). The dataframe above shows the actual unit of analysis: each row is a single content page, with its visibility (`impressions_90d`), staleness (`days_since_last_update`), position, and the target proxy (`is_declining_label`) all sitting side by side. This is the grain the model will score and rank — not client, not query, not day.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "word_count", "trend_direction"]
unit_df = df[cols].copy()
unit_df["is_declining_label"] = (unit_df["trend_direction"].str.lower() == "down").astype(int)

print(f"Rows (pages): {unit_df.shape[0]}")
unit_df.head(10)
print(unit_df["is_declining_label"].value_counts())

Rows (pages): 30000
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule (e.g. "stale AND visible") can only combine a small number of hand-chosen thresholds the way a human already guessed they should combine. Week 2 showed this rule scored very well in-sample (Precision@20 = 0.900) but a decision tree  which can find interactions a human wouldn't think to hand-code still found different, useful splits (its first split was `impressions_90d`, not staleness). More importantly, only a model validated on holdout data reveals whether a rule's apparent strength is real or an artifact of the specific pages it was tuned on. A model gives you a probability you can rank by and honestly test against unseen clients, which a static if/else cannot do.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.